In [4]:
import pandas as pd
import psycopg2
import os
from pathlib import Path
from dotenv import load_dotenv
from Levenshtein import distance as levenshtein_distance
import matplotlib.pyplot as plt
from collections import Counter
import time

# for clustering
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score


In [5]:
notebook_dir = Path.cwd()
root_dir = notebook_dir.parent

dotenv_path = root_dir / ".env"
load_dotenv(dotenv_path)

DB_USER = os.environ.get("DB_USER")
DB_PASS = os.environ.get("DB_PASS")
DB_NAME = os.environ.get("DB_NAME")
DB_HOST = os.environ.get("DB_HOST")
DB_PORT = os.environ.get("DB_PORT")

WD_STRING_TYPES = ['monolingualtext', 'string', 'external-id', 'url', 'commonsMedia', 'geo-shape', 'tabular-data', 'math', 'musical-notation', 'unknown-values']
WD_ENTITY_TYPES = ['wikibase-item', 'wikibase-entityid', 'wikibase-property', 'wikibase-lexeme', 'wikibase-sense', 'wikibase-form', 'entity-schema']

DATA_FILE_PATH = str(root_dir) + '/data' + '/changes_for_clustering_string_updates.parquet'

In [ ]:
import json
from datetime import datetime

class ExperimentTracker:
    def __init__(self, experiments_dir='experiments'):
        self.experiments_dir = Path(experiments_dir)
        self.experiments_dir.mkdir(exist_ok=True)
        self.timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.experiment_dir = self.experiments_dir / self.timestamp
        self.experiment_dir.mkdir(exist_ok=True)
        
    def log_params(self, params):
        """Log experiment parameters"""
        with open(self.experiment_dir / 'params.json', 'w') as f:
            json.dump(params, f, indent=2, default=str)
    
    def log_metrics(self, metrics):
        """Log experiment metrics"""
        with open(self.experiment_dir / 'metrics.json', 'w') as f:
            json.dump(metrics, f, indent=2)
    
    def save_results(self, df, name='results'):
        """Save results DataFrame"""
        df.to_parquet(self.experiment_dir / f'{name}.parquet', compression='snappy')
    
    def save_model(self, model, scaler, label_encoders):
        """Save model artifacts"""
        import pickle
        with open(self.experiment_dir / 'model.pkl', 'wb') as f:
            pickle.dump({'model': model, 'scaler': scaler, 'encoders': label_encoders}, f)
    
    def save_figure(self, fig, name):
        """Save matplotlib figure"""
        fig.savefig(self.experiment_dir / f'{name}.png', dpi=300, bbox_inches='tight')
    
    def save_text(self, text, name):
        """Save text output"""
        with open(self.experiment_dir / f'{name}.txt', 'w') as f:
            f.write(text)
    
    def get_summary(self):
        """Get path to this experiment"""
        return str(self.experiment_dir)

## Clustering

In [7]:
def perform_clustering(X, method='kmeans', n_clusters=10, **kwargs):
    """
    Perform clustering with different methods
    X is already scaled
    """
    
    if method == 'kmeans':
        # random state guarantees that the results are reproducible
        model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        labels = model.fit_predict(X)
        
    elif method == 'dbscan':
        eps = kwargs.get('eps', 0.5)
        min_samples = kwargs.get('min_samples', 5)
        model = DBSCAN(eps=eps, min_samples=min_samples)
        labels = model.fit_predict(X)
    
    return labels, X

def find_optimal_k(X_scaled, tracker, max_k=20, save_plot=True):
    """
    Use elbow method to find optimal number of clusters
    """
    inertias = []
    silhouette_scores = []
    K = range(2, max_k + 1)
    
    print(f"Testing k from 2 to {max_k}...")
    for k in K:
        # K-means is a centroid-based clustering algorithm, where we calculate the distance between each data point and a centroid to assign it to a cluster. 
        # The goal is to identify the K number of groups in the dataset.
        # It is iterative and minimizes distances between the data points and the cluster centroid.
        # After assigning points to clusters, the centroids are recalculated as the mean of the assigned points, and the process repeats until convergence.

        print(f"k={k}...", end=' ')
        kmeans = KMeans(n_clusters=k, random_state=20, n_init=20) 
        # n_init states how many different sets of randomly chosen centroids the algorithm should use
        # For each different set of points, a comparision is made about how much distance did the clusters move
        kmeans.fit(X_scaled)
        inertias.append(kmeans.inertia_)
        
        # Calculate silhouette score
        labels = kmeans.labels_
        sil_score = silhouette_score(X_scaled, labels, metric = 'euclidean')
        silhouette_scores.append(sil_score)
        print(f"Silhouette: {sil_score:.3f}")
    
    # Plot both metrics
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Elbow curve
    ax1.plot(K, inertias, 'bx-', linewidth=2, markersize=8)
    ax1.set_xlabel('Number of clusters (k)', fontsize=12)
    ax1.set_ylabel('Inertia', fontsize=12)
    ax1.set_title('Elbow Method For Optimal k', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Silhouette scores
    ax2.plot(K, silhouette_scores, 'go-', linewidth=2, markersize=8)
    ax2.set_xlabel('Number of clusters (k)', fontsize=12)
    ax2.set_ylabel('Silhouette Score', fontsize=12)
    ax2.set_title('Silhouette Score (Higher = Better)', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Mark best silhouette -> the k with the highest silhouette score
    best_k = K[silhouette_scores.index(max(silhouette_scores))]
    ax2.axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
    ax2.legend()
    
    plt.tight_layout()
    
    if save_plot:
        plt.savefig(tracker.experiments_dir + '/' + 'elbow_analysis.png', dpi=300, bbox_inches='tight')
        print(f"\nSaved plot to elbow_analysis.png")
    
    plt.show()
    
    print("\n" + "="*50)
    print(f"BEST K (by Silhouette Score): {best_k}")
    print("="*50)
    
    # Save metrics
    metrics_df = pd.DataFrame({
        'k': list(K),
        'inertia': inertias,
        'silhouette_score': silhouette_scores
    })
    metrics_df.to_csv(tracker.experiments_dir + '/' +'clustering_metrics.csv', index=False)
    print("Saved metrics to clustering_metrics.csv")
    
    return K, inertias, silhouette_scores, best_k


def visualize_clusters(X_scaled, labels, method='pca'):
    """
    Visualize clusters in 2D
    """
    if method == 'pca':
        reducer = PCA(n_components=2, random_state=42)
        X_2d = reducer.fit_transform(X_scaled)
        title = f'PCA Visualization (explained var: {reducer.explained_variance_ratio_.sum():.2%})'
        
    elif method == 'tsne':
        reducer = TSNE(n_components=2, random_state=42, perplexity=30)
        X_2d = reducer.fit_transform(X_scaled)
        title = 't-SNE Visualization'
    
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='tab20', alpha=0.6)
    plt.colorbar(scatter)
    plt.title(title)
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')
    plt.show()


def analyze_clusters(df, labels, tracker=None, output_file_examples='cluster_examples.csv', output_file_analysis='cluster_analysis.csv', n_examples=20, datatype='string'):
    """
    Enhanced cluster analysis with detailed statistics
    """
    df_with_clusters = df.copy()
    df_with_clusters['cluster'] = labels
    
    from io import StringIO
    output = StringIO()
     
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    
    # Overall statistics
    header = f"CLUSTERING ANALYSIS - {n_clusters} clusters\n"
    header += "=" * 80 + "\n\n"
    header += f"Total samples: {len(df):,}\n"
    header += f"Cluster sizes:\n"
    
    cluster_sizes = Counter(labels)
    for cluster_id in sorted(cluster_sizes.keys()):
        size = cluster_sizes[cluster_id]
        pct = size / len(df) * 100
        header += f"  Cluster {cluster_id}: {size:,} ({pct:.1f}%)\n"
    
    print(header)
    output.write(header + "\n")
    
    for cluster_id in sorted(set(labels)):
        if cluster_id == -1:
            continue
        
        cluster_data = df_with_clusters[df_with_clusters['cluster'] == cluster_id]
        cluster_output = f"\n{'='*80}\n"
        cluster_output += f"CLUSTER {cluster_id} (n={len(cluster_data):,}, {len(cluster_data)/len(df)*100:.1f}%)\n"
        cluster_output += f"{'='*80}\n\n"
        
        # User type distribution
        cluster_output += "USER TYPE:\n"
        for user_type, count in cluster_data['user_type'].value_counts().items():
            cluster_output += f"  {user_type}: {count:,} ({count/len(cluster_data)*100:.1f}%)\n"

        # Datatype distribution
        cluster_output += "\nDATATYPE:\n"
        for dtype, count in cluster_data['datatype'].value_counts().items():
            cluster_output += f"  {dtype}: {count:,} ({count/len(cluster_data)*100:.1f}%)\n"
        
        # Action distribution
        cluster_output += "\nACTION:\n"
        for action, count in cluster_data['action'].value_counts().items():
            cluster_output += f"  {action}: {count:,} ({count/len(cluster_data)*100:.1f}%)\n"

        # Top properties
        cluster_output += "\nTOP 10 PROPERTIES:\n"
        top_props = cluster_data['property_id'].value_counts().head(10)
        for prop, count in top_props.items():
            if 'property_label' in cluster_data.columns:
                label = cluster_data['property_label'].iloc[0]
                cluster_output += f"  {prop} ({label}): {count:,} ({count/len(cluster_data)*100:.1f}%)\n"
            else:
                cluster_output += f"  {prop}: {count:,} ({count/len(cluster_data)*100:.1f}%)\n"
        
        # Change magnitude 
        if 'levenshtein_distance' in cluster_data.columns:
            actual_mag = cluster_data['levenshtein_distance']
            if len(actual_mag) > 0:
                cluster_output += f"\nLEVENSHTEIN DISTANCE:\n"
                cluster_output += f"  Mean: {actual_mag.mean():.2f}\n"
                cluster_output += f"  Range: [{actual_mag.min():.2f}-{actual_mag.max():.2f}]\n"
        
        print(cluster_output)
        output.write(cluster_output + "\n")

    
    # Save to tracker if provided
    if tracker:
        tracker.save_text(output.getvalue(), tracker.experiments_dir + '/' + output_file_analysis)

    # Get examples
    output_rows = []
    
    for cluster_id in sorted(set(labels)):
        cluster = df_with_clusters[df_with_clusters['cluster'] == cluster_id]
        
        if len(cluster) == 0:
            continue
        
        # Get examples
        example_cols = ['revision_id', 'entity_id', 'entity_label', 'property_id', 'property_label', 
                       'old_value', 'new_value', 'old_value_label', 'new_value_label', 
                       'user_type', 'datatype', 'change_target', 'value_id', 'action', 'target', 'timestamp']
        example_cols = [col for col in example_cols if col in cluster.columns]
        
        # Sample examples
        examples = cluster[example_cols].sample(min(n_examples, len(cluster)), random_state=42)
        
        for idx, row in examples.iterrows():
            output_row = {
                'cluster_id': cluster_id,
                'cluster_size': len(cluster)
            }
            
            # Add example data
            for col in example_cols:
                output_row[col] = row[col]
            
            output_rows.append(output_row)
    
    # Convert to DataFrame and save
    results_df = pd.DataFrame(output_rows)
    results_df.to_csv(tracker.experiments_dir + '/' + output_file_examples, index=False)
    print(f"\n Saved {len(results_df)} examples to {output_file_examples}")

    return results_df



## General features

In [ ]:
def extract_general_change_features(df, feature_cols):
    """
    General features
    """
    df = df.copy()

    label_encoders = {}
    categorical_cols = [
        'user_type',  # bot/human/anonymous
        # 'datatype',
        # 'action'
    ]
    
    for col in categorical_cols:
        if col in df.columns:
            le = LabelEncoder()
            df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
            label_encoders[col] = le

    feature_cols.extend([
        # 'datatype_encoded',
        'user_type_encoded',
        # 'action_encoded'
    ])

    return df, feature_cols

## Time features

In [ ]:
def create_time_features(df, feature_cols):
    """
    Extract time change features
    """
    time_mask = (df['datatype'] == 'time')
    
    if time_mask.sum() == 0:
        return df
    
    def parse_wikidata_time(time_str):
        try:
            time_str = str(time_str).lstrip('+')
            return pd.to_datetime(time_str, format='%Y-%m-%dT%H:%M:%SZ')
        except:
            return pd.NaT
    
    df.loc[time_mask, 'old_time_parsed'] = df.loc[time_mask, 'old_value'].apply(parse_wikidata_time)
    df.loc[time_mask, 'new_time_parsed'] = df.loc[time_mask, 'new_value'].apply(parse_wikidata_time)
    
    valid_mask = time_mask & df['old_time_parsed'].notna() & df['new_time_parsed'].notna()
    
    if valid_mask.sum() == 0:
        return df
    
    try:
        df.loc[valid_mask, 'time_diff_days'] = (
            df.loc[valid_mask, 'new_time_parsed'] - df.loc[valid_mask, 'old_time_parsed']
        ).dt.days.abs()
    except (OverflowError, ValueError):
        # Fallback: calculate manually using timestamps (seconds since epoch)
        old_timestamps = df.loc[valid_mask, 'old_time_parsed'].astype('int64') / 10**9  # Convert to seconds
        new_timestamps = df.loc[valid_mask, 'new_time_parsed'].astype('int64') / 10**9
        df.loc[valid_mask, 'time_diff_days'] = ((new_timestamps - old_timestamps) / 86400).abs()
    
    df.loc[valid_mask, 'time_diff_years'] = df.loc[valid_mask, 'time_diff_days'] / 365.25

    df.loc[valid_mask, 'levenshtein_distance'] = df.loc[valid_mask].apply(
        lambda row: levenshtein_distance(row['old_value'], row['new_value']), 
        axis=1
    )

    feature_cols.extend([
        'time_diff_days',
        'time_diff_years',
        'levenshtein_distance'
    ])

    return df, feature_cols


## Globe cordinate features

In [ ]:
def create_globe_coordinate_features(df, feature_cols):
    coordinate_mask = df['datatype'] == 'globecoordinate'
    
    for idx in df[coordinate_mask].index:
        try:
            old_val = json.loads(df.loc[idx, 'old_value'])
            new_val = json.loads(df.loc[idx, 'new_value'])
            
            df.loc[idx, 'latitude_diff_abs'] = abs(new_val['latitude'] - old_val['latitude'])
            df.loc[idx, 'longitude_diff_abs'] = abs(new_val['longitude'] - old_val['longitude'])
            
            # distance with haversine formula
            from math import radians, sin, cos, sqrt, atan2
            
            lat1, lon1 = radians(old_val['latitude']), radians(old_val['longitude'])
            lat2, lon2 = radians(new_val['latitude']), radians(new_val['longitude'])
            
            dlat = lat2 - lat1
            dlon = lon2 - lon1
            
            a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
            c = 2 * atan2(sqrt(a), sqrt(1-a))
            distance_km = 6371 * c  # Earth radius in km
            
            df.loc[idx, 'coordinate_distance_km'] = distance_km

        except:
            pass

    feature_cols.extend([
        'latitude_diff_abs',
        'longitude_diff_abs',
        'coordinate_distance_km'
    ])

    return df, feature_cols

## Quantity features

In [ ]:
def create_quantity_features(df, feature_cols):
    quant_mask = df['datatype'] == 'quantity'
    
    if quant_mask.sum() == 0:
        return df, feature_cols

    subset = df[quant_mask].copy()
    
    # remove + sign
    subset['old_str'] = subset['old_value'].astype(str).str.replace('+', '', regex=False)
    subset['new_str'] = subset['new_value'].astype(str).str.replace('+', '', regex=False)
    
    subset['old_float'] = pd.to_numeric(subset['old_str'], errors='coerce')
    subset['new_float'] = pd.to_numeric(subset['new_str'], errors='coerce')
    
    valid_mask = subset['old_float'].notna() & subset['new_float'].notna()
    
    # value diff absolute
    subset.loc[valid_mask, 'value_diff_abs'] = (
        subset.loc[valid_mask, 'new_float'] - subset.loc[valid_mask, 'old_float']
    ).abs()
    
    # sign change 
    subset.loc[valid_mask, 'sign_change'] = (
        subset.loc[valid_mask, 'old_float'] * subset.loc[valid_mask, 'new_float'] < 0
    ).astype(int)
    
    # precision difference 
    def get_decimal_places(val_str):
        val_str = str(val_str)
        if '.' in val_str:
            decimal_part = val_str.split('.')[1] if len(val_str.split('.')) > 1 else ''
            return len(decimal_part)
        elif ',' in val_str:
            decimal_part = val_str.split(',')[1] if len(val_str.split(',')) > 1 else ''
            return len(decimal_part)
        else:
            return 0
    
    subset.loc[valid_mask, 'old_decimal_places'] = subset.loc[valid_mask, 'old_str'].apply(get_decimal_places)
    subset.loc[valid_mask, 'new_decimal_places'] = subset.loc[valid_mask, 'new_str'].apply(get_decimal_places)
    
    subset.loc[valid_mask, 'precision_diff_abs'] = (
        subset.loc[valid_mask, 'new_decimal_places'] - subset.loc[valid_mask, 'old_decimal_places']
    ).abs()
    
    # add changes to original df
    df.loc[quant_mask, 'value_diff_abs'] = subset['value_diff_abs']
    df.loc[quant_mask, 'sign_change'] = subset['sign_change']
    df.loc[quant_mask, 'precision_diff_abs'] = subset['precision_diff_abs']
    
    feature_cols.extend([
        'value_diff_abs',
        'sign_change',
        'precision_diff_abs'
    ])
    
    return df, feature_cols

## Text features

Useful to use ratios for levenshtein distance because they give you a percentage of what changed.

Example 1:
- Old: "cat"
- New: "dog"
- Levenshtein distance: 3

Without ratio: distance = 3 

With ratio: 3 / 3 = 1 (100% changed)

Example 2:
- Old: "The quick brown fox jumps over the lazy dog"
- New: "The quick brown fox jumps over the lazy cat"
- Levenshtein distance: 3 (dog -> cat)

Without ratio: distance = 3 (same as before)

With ratio: 3 / 44 = 0.068 (only 6.8% changed)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from Levenshtein import distance as levenshtein_distance
import os
import multiprocessing as mp

def extract_text_features(df, old_col, new_col, mask):

    if mask.sum() == 0:
        return df
    
    subset = df.loc[mask].copy()
    
    subset['old'] = subset[old_col].astype(str)
    subset['new'] = subset[new_col].astype(str)
    
    valid = (subset['old'] != '{}') & (subset['new'] != '{}') & (subset['old'] != 'nan') & (subset['new'] != 'nan')
    subset = subset[valid]
    
    if len(subset) == 0:
        return df
    
    subset['length_diff_abs'] = (subset['new'].str.len() - subset['old'].str.len()).abs()
    
    subset['case_differs'] = (
        (subset['old'] != subset['new']) & 
        (subset['old'].str.lower() == subset['new'].str.lower())
    ).astype(int)
    
    old_no_space = subset['old'].str.replace(' ', '', regex=False)
    new_no_space = subset['new'].str.replace(' ', '', regex=False)
    subset['spaces_differs'] = (
        (subset['old'] != subset['new']) & 
        (old_no_space == new_no_space)
    ).astype(int)
    
    old_no_punct = subset['old'].str.replace(r'[^\w\s]', '', regex=True).str.replace(' ', '', regex=False)
    new_no_punct = subset['new'].str.replace(r'[^\w\s]', '', regex=True).str.replace(' ', '', regex=False)
    subset['punct_differs'] = (
        (subset['old'] != subset['new']) & 
        (old_no_punct == new_no_punct)
    ).astype(int)
    
    old_no_dash = subset['old'].str.replace(r'[-–—_]', '', regex=True).str.replace(' ', '', regex=False)
    new_no_dash = subset['new'].str.replace(r'[-–—_]', '', regex=True).str.replace(' ', '', regex=False)
    subset['hyph_dash_differs'] = (
        (subset['old'] != subset['new']) & 
        (old_no_dash == new_no_dash)
    ).astype(int)
    
    old_no_brackets = subset['old'].str.replace(r'[\[\]\(\)\{\}"\']', '', regex=True).str.replace(' ', '', regex=False)
    new_no_brackets = subset['new'].str.replace(r'[\[\]\(\)\{\}"\']', '', regex=True).str.replace(' ', '', regex=False)
    subset['brackets_differs'] = (
        (subset['old'] != subset['new']) & 
        (old_no_brackets == new_no_brackets)
    ).astype(int)
    
    subset['token_count_old'] = subset['old'].str.split().str.len()
    subset['token_count_new'] = subset['new'].str.split().str.len()
    
    def calc_overlap(row):
        old_tokens = set(row['old'].split())
        new_tokens = set(row['new'].split())
        if len(old_tokens | new_tokens) == 0:
            return 0
        return len(old_tokens & new_tokens) / len(old_tokens | new_tokens)
    
    # percentage (ratio) of token overlap
    subset['token_overlap'] = subset.apply(calc_overlap, axis=1)
    
    subset['old_in_new'] = subset.apply(lambda row: int(row['old'] in row['new']), axis=1)
    subset['new_in_old'] = subset.apply(lambda row: int(row['new'] in row['old']), axis=1)
        
    subset['levenshtein_distance'] = subset.apply(
        lambda row: levenshtein_distance(row['old'].lower().strip(), row['new'].lower().strip()),
        axis=1
    )

    old_len = subset['old'].str.len()
    new_len = subset['new'].str.len()
    max_len = np.maximum(old_len, new_len).clip(lower=1) # replace 0's with 1 to avoid division by zero

    # percentage of how much changed
    subset['edit_distance_ratio'] = subset['levenshtein_distance'] / max_len
    
    feature_cols = ['levenshtein_distance', 'length_diff_abs', 'case_differs', 
                'spaces_differs', 'punct_differs', 'hyph_dash_differs', 
                'brackets_differs', 'token_count_old', 'token_count_new', 
                'token_overlap', 'old_in_new', 'new_in_old', 'edit_distance_ratio']

    for col in feature_cols:
        df.loc[subset.index, col] = subset[col]
    
    return df, feature_cols


def create_semantic_similarity_features(df, old_col, new_col, feature_cols, string_mask):
    """
    Calculates cosine similarity between old and new value embeddings
    """
    
    if string_mask.sum() == 0:
        return df, feature_cols
    
    subset = df[string_mask].copy()
    
    old_texts = subset[old_col].astype(str).tolist()
    new_texts = subset[new_col].astype(str).tolist()
    
    print(f"Encoding {len(old_texts)} text pairs using multiple CPU cores...")
    
    # Limit number of cpuS to use
    num_workers = 5

    # load model
    model = SentenceTransformer('all-MiniLM-L6-v2')
    pool = model.start_multi_process_pool(target_devices=[f'cpu:{i}' for i in range(num_workers)])
    
    old_embeddings = model.encode_multi_process(
        old_texts,
        pool,
        batch_size=256,
        chunk_size=1000,
        show_progress_bar=True
    )
    
    new_embeddings = model.encode_multi_process(
        new_texts,
        pool,
        batch_size=256,
        chunk_size=1000,
        show_progress_bar=True
    )
    
    model.stop_multi_process_pool(pool)

    # calculate cosine similarity
    similarities = np.array([
        cosine_similarity([old_emb], [new_emb])[0][0]
        for old_emb, new_emb in zip(old_embeddings, new_embeddings)
    ])
    
    subset['semantic_similarity'] = similarities
    
    # update original df
    df.loc[string_mask, 'semantic_similarity'] = subset['semantic_similarity']
    
    feature_cols.append('semantic_similarity')
    
    return df, feature_cols

def create_text_features(df, feature_cols):
    """Extract text features for string datatypes"""
    string_mask = df['datatype'].isin(WD_STRING_TYPES)
    
    print(f"Creating basic features.")
    start_time = time.time()
    df, feature_cols = extract_text_features(df, 'old_value', 'new_value', string_mask)
    end_time = time.time()
    print(f"Basic features created in {end_time - start_time:.2f} seconds.")
    
    print('Creating semantic similarity from embeddings.')
    start_time = time.time()
    df, feature_cols = create_semantic_similarity_features(df, 'old_value', 'new_value', feature_cols, string_mask)
    end_time = time.time()
    print(f"Semantic similarity features created in {end_time - start_time:.2f} seconds.")

    df, feature_cols = extract_general_change_features(df, feature_cols)

    return df, feature_cols

## Entity features

In [10]:
def create_entity_features(df, feature_cols):
    """Extract features for entity datatypes using labels"""
    entity_mask = df['datatype'].isin(WD_ENTITY_TYPES)
    
    print(f"Processing {entity_mask.sum()} entity changes.")
    df, feature_cols = extract_text_features(df, 'old_value_label', 'new_value_label', entity_mask)

    print('Creating semantic similarity from embeddings.')
    start_time = time.time()
    df, feature_cols = create_semantic_similarity_features(df, 'old_value_label', 'new_value_label', feature_cols, entity_mask)
    end_time = time.time()
    print(f"Semantic similarity features created in {end_time - start_time:.2f} seconds.")
    
    return df, feature_cols

In [ ]:

# TODO : REMOVE
def prepare_features(df, datatype):
    """
        Creates features depending on the datatype 
    """
    
    feature_cols = []

    # Extract general features (independent of datatype)
    df, feature_cols = extract_general_change_features(df, feature_cols)

    if datatype == 'quantity':
        df, feature_cols = create_quantity_features(df, feature_cols)
    elif datatype == 'time':
        df, feature_cols = create_time_features(df, feature_cols)
    elif datatype == 'globecoordinate':
        df, feature_cols = create_globe_coordinate_features(df, feature_cols)
    elif datatype == 'string':
        df, feature_cols = create_text_features(df, feature_cols)
    elif datatype in 'entity':
        df, feature_cols = create_entity_features(df, feature_cols)
    
    feature_cols = [col for col in feature_cols if col in df.columns]
    X = df[feature_cols].fillna(0)
    
    return X, df, feature_cols

## Auxiliary methods

In [11]:
import matplotlib.pyplot as plt

def analyze_feature_distributions(X):

    for col in X.columns:
        print(f"\n{col} statistics:")
        print(X[col].describe())
        print(f"Min: {X[col].min()}, Max: {X[col].max()}")
        
        # Plot histogram
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.hist(X[col].dropna(), bins=50)
        plt.title(f'{col} - Original')
        plt.xlabel('Value')
        
        # Plot boxplot to see outliers
        plt.subplot(1, 2, 2)
        plt.boxplot(X[col].dropna())
        plt.title(f'{col} - Boxplot')
        plt.tight_layout()
        plt.show()

## Cluster string changes

In [ ]:
def cluster_string_changes(change_target='value'):
    df = pd.read_parquet(DATA_FILE_PATH)

    if change_target == 'value':
        df_updates = df[(df['action'] == 'UPDATE') & (df['change_target'] == '')].copy()
    elif change_target == 'datatype_metaddata':
        df_updates = df[(df['action'] == 'UPDATE') & (df['change_target'] != '') & (df['change_target'] != 'rank')].copy()
    else: # rank updates don't make much sense to analyze
        return
    
    if change_target == 'value':
        df_updates = df_updates[df_updates['datatype'] == 'string'].copy()
    
    print(f"Total {change_target} updates: {len(df_updates):,}")
     
    # get features
    print('Prepare features')
    df_updates, feature_cols = create_text_features(df_updates, [])
    X = df_updates[feature_cols].fillna(0)

    X = X.astype(float).values  # df to numpy array

    print(X.mean(axis=0))

    # scale so std is ~1 and mean ~0
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    print('Mean after scaling: ', X_scaled.mean(axis=0))
    print('Std after scaling: ', X_scaled.std(axis=0))

    # PCA should make mean ~0
    pca = PCA(n_components=0.95) 
    X_reduced = pca.fit_transform(X_scaled)
    print('Mean after PCA: ', X_reduced.mean(axis=0))
    print(f"\nReduced from {X_scaled.shape[1]} to {X_reduced.shape[1]} features")

    # Find optimal k
    print('\n' + "="*50)
    print("FINDING OPTIMAL K")
    print("="*50)
    K, inertias, silhouette_scores, best_k = find_optimal_k(X_reduced, max_k=20)
    
    print(f"Best k determined: {best_k}")
    k = best_k  
    print(f"\nUsing k={k} for clustering...")
    
    # Cluster
    print('Clustering...')
    labels, _ = perform_clustering(X_reduced, n_clusters=k)
    
    tracker = ExperimentTracker()
    # Analyze and save examples to CSV
    results_df = analyze_clusters(
        df_updates, 
        labels, 
        tracker,
        output_file_examples=f'cluster_examples_string.csv',
        output_file_analysis=f'cluster_analysis_string.csv',
        n_examples=20
    )
    
    print(f"Saved examples to cluster_examples_string.csv")

In [ ]:
change_target = 'value'
datatype = 'string'
df = pd.read_parquet(DATA_FILE_PATH)

if change_target == 'value':
    df_updates = df[(df['action'] == 'UPDATE') & (df['change_target'] == '')].copy()
elif change_target == 'datatype_metaddata':
    df_updates = df[(df['action'] == 'UPDATE') & (df['change_target'] != '') & (df['change_target'] != 'rank')].copy()


if change_target == 'value':
    df_updates = df_updates[df_updates['datatype'] == 'string'].copy()

print(f"Total {change_target} updates: {len(df_updates):,}")
    
print('Creating features')
df_updates, feature_cols = create_text_features(df_updates, [])
X = df_updates[feature_cols].fillna(0)

# Save calculated features
X.to_parquet('string_change_features.parquet', compression='snappy')

# X = X.astype(float).values  # df to numpy array

# print(X.mean(axis=0))

# analyze_feature_distributions(pd.DataFrame(X, columns=feature_cols))

Total value updates: 494,396
Creating features
Creating basic features.
Basic features created in 56.17 seconds.
Creating semantic similarity from embeddings.
Encoding 494396 text pairs using multiple CPU cores...


/tmp/ipykernel_888967/1974824336.py:120: DeprecationWarning: The `encode_multi_process` method has been deprecated, and its functionality has been integrated into `encode`. You can now call `encode` with the same parameters to achieve multi-process encoding.
  old_embeddings = model.encode_multi_process(


Chunks:   0%|          | 0/495 [00:00<?, ?it/s]

/tmp/ipykernel_888967/1974824336.py:128: DeprecationWarning: The `encode_multi_process` method has been deprecated, and its functionality has been integrated into `encode`. You can now call `encode` with the same parameters to achieve multi-process encoding.
  new_embeddings = model.encode_multi_process(


Chunks:   0%|          | 0/495 [00:00<?, ?it/s]